# Tensores

### 1. Qué es un Tensor
- Estructura similar a las matrices y a las arrays de Numpy por lo tanto son compatibles
- Pueden correr en CPU o GPU

In [ ]:
import torch
import numpy as np

### Como inicializar un Tensor

Se pueden inicializar de variaas maneras

**Desde datos directamente**
- A partir de listas o valores
- El tipo de dato dtype se infiere automaticamente

In [ ]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

**Desde un array de Numpy**
- se puede crear un tensor a partir de ndarray
- se puede convertir un tensor a numpy directamente

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

**Desde otro tensor**

- El nuevo tensor hereda propiedades del tensor original: forma, tipo de dato.

- Se pueden sobrescribir explícitamente si se desea.

In [ ]:
x_ones = torch.ones_like(x_data) # retiene las propiedades de x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # sobreescribe el tipo de dato de x_data
print(f"Random Tensor: \n {x_rand} \n")

**Inicialización con valores aleatorios o constantes**

Se pueden crear tensores con ceros, unos o valores aleatorios, especificando la forma (shape) como una tupla.

Ejemplos:

In [ ]:
shape = (2, 3)  # 2 filas, 3 columnas

zeros = torch.zeros(shape)       # todos ceros
ones = torch.ones(shape)         # todos unos
rand = torch.rand(shape)         # valores aleatorios entre 0 y 1
randn = torch.randn(shape)       # valores aleatorios con distribución normal (media=0, var=1)
full = torch.full(shape, 7)      # todos con valor 7


### Atributos de un Tensor

- x.shape → devuelve las dimensiones del tensor

- x.size() → lo mismo que x.shape

- x.dtype → tipo de datos (torch.float32, torch.int64, etc.)

- x.device → dispositivo donde está el tensor (cpu o cuda)

- x.requires_grad → si el tensor está marcado para calcular gradientes

Ejemplo:

In [ ]:
x = torch.rand(2, 3, dtype=torch.float32, device='cuda', requires_grad=True)
print(x.shape)         # torch.Size([2, 3])
print(x.dtype)         # torch.float32
print(x.device)        # cuda:0
print(x.requires_grad) # True

### Operaciones avanzadas con Tensores
- Se puede hacer con @ o matmul().
- También se puede guardar el resultado en un tensor preexistente usando out=.

In [ ]:
y1 = tensor @ tensor.T         # operador @
y2 = tensor.matmul(tensor.T)   # método matmul
y3 = torch.rand_like(y1)       
torch.matmul(tensor, tensor.T, out=y3)  # resultado guardado en y3

Producto elemento a elemento (Hadamard product)

- Se puede hacer con * o mul().

- También admite out= para escribir el resultado en un tensor existente.

In [ ]:
z1 = tensor * tensor
z2 = tensor.mul(tensor)
z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

Agregación y conversión a valor de Python

- Para sumar todos los elementos y obtener un número de Python:

In [ ]:
agg = tensor.sum()
agg_item = agg.item()  # convierte a float/int
print(agg_item, type(agg_item))

Operaciones in-place

- Modifican el tensor original directamente (nota el _ al final del método):

In [ ]:
tensor.add_(5)  # suma 5 a todos los elementos de tensor en su lugar

# Datasets

### Creacion de un Dataset a partir de datos

- Para crear un dataset propio, tu clase debe heredar de torch.utils.data.Dataset y definir tres métodos clave:

__init__(self, ...)

- Se ejecuta al crear el objeto.

- Sirve para cargar rutas de archivos, etiquetas, transformaciones, etc.

__len__(self)

- Devuelve el número total de muestras en el dataset.

- Permite que PyTorch sepa cuántos elementos hay.

__getitem__(self, idx)

- Devuelve la muestra y su etiqueta en la posición idx.

Aquí se pueden aplicar transformaciones o preprocesamiento a la muestra.

In [ ]:
import os
import pandas as pd
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

### Preparar los datos para entrenar con DataLoaders

- Un Dataset devuelve una muestra a la vez (feature y etiqueta).

- Para entrenamiento, necesitamos:

- Minibatches: pasar varias muestras a la vez para eficiencia y estabilidad en el entrenamiento.

- Barajar los datos (shuffle): evita que el modelo aprenda el orden de los datos y reduce overfitting.

- Multiprocesamiento (num_workers): acelera la carga de datos usando varios procesos de Python.

- DataLoader abstrae toda esta complejidad y permite acceder a los datos fácilmente en un bucle de entrenamiento.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

### Iterar a traves del DataLoader
- Cada iteración devuelve un minibatch de features y labels.

- El tamaño del minibatch se define con batch_size al crear el DataLoader.

- Si shuffle=True, los datos se barajan automáticamente al completar una pasada (epoch).

- Para control más avanzado del orden de carga, se pueden usar Samplers.

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

# Transforms
- Los datos no siempre vienen en el formato que necesitan los modelos.

- Transforms permiten modificar features y labels para que sean adecuados para entrenamiento.

- Parámetros clave en datasets de TorchVision

- transform → transforma las features (ej. imágenes)

- target_transform → transforma las labels (ej. one-hot encoding)

- Transformaciones comunes

- Usando el módulo torchvision.transforms:

- ToTensor → convierte imágenes PIL o NumPy a tensores.

- Lambda → aplica cualquier función personalizada sobre features o labels.

- Se pueden encadenar varias transformaciones con transforms.Compose([...]).

In [ ]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

### Ejemplo con lambda

In [ ]:
target_transform = Lambda(lambda y: torch.zeros(
    10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))

# Construyendo la red neuronal

-Cada modulo de Pytorch es una subclase del modulo nn, una red neuronal es un modeulo que consiste en otros modulos(layers). 


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### Definir la clase

Definimos nuestra red nueronal con la subcalse nn.Module

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

- Para poder usar el modelo le pasamos los datos de entrada, eso ejecuta el modelo forward de fondo no hay que llamar al forward directamente.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

---

### Model layers
para ilustrar el modelo de MNIST cogeremos un minibatch de 3 img con dimensiones de 28x28 y veremos qie [asa a traves de la network]

In [ ]:
input_image = torch.rand(3,28,28)
print(input_image.size())

### nn.Flatten
Inicializaremos el nn.Flatten leyer para convertir cada imagen 2d de 28x28 a un array de 784 vpix

In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

### nn.ReLu
En este ejemplo usaremos nn.ReLU en medio de nuestras linear layers, pero hay otras activaciones para introducir no linearidad en el modelo

In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

### nn.Sequential

nn.Sequential es nun contenedor ordenado de modulos. Los datos pasan mediante todos los modulos en el mismo orden en el que se definen. Puedes usar contenedores secuenciales para poner junto a una neurona

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

### nn.Softmax

 La ultima capa lineal de la red neuronal devuelve [logits]{.title-ref}
- 

In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

### Parmetros del modelo
La mayoria de capas dentro de la red neuronal estan parametrizadas, tienen asociados pesos y bias asociados que se optimizan durante el entrenamiento, subclasificar nn.Module rastrea automaticamente los campos definidos dentro de su objeto modelo y hace que todos los parametros sean accecibles usando parameters().
En el ejemplo iteramos cada parametro y despues imprimieremos su tamano y una vista previa de sus valores

In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

# Diferenciacion automatica con torch.autograd

- Cuando entrenamos redes neuronales, el algoritmo mas frucuente es el backpropagation donde los pesos se ajustan por el gradiente de la funcon de perdida respecto al parametro dado.
- Para calcular los gradientes, pytorch usa torch.autograd y soporta la el calculo del gradiente para cualquier grafo computacional.
- Considerenato una simple red neuronal de una capa, con una entrada x y parametros como w y b y una funcoin de perdida se puede puede definir en pytorch de esta manera.

Tensors, Functions and Computational graph
==========================================

This code defines the following **computational graph**:

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

In this network, `w` and `b` are **parameters**, which we need to
optimize. Thus, we need to be able to compute the gradients of loss
function with respect to those variables. In order to do that, we set
the `requires_grad` property of those tensors.

aUna funcoin que aplicamos a tensores para construir un grado computaciones es un objeto de la clase 'Function'.
- El objeto sabe como calcular la funcion tanto como el forward como el backward.
- U"na referencia a la funcoin de la backpropagation se almacena en la varible grad_fn de un tensor.

In [ ]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

## Calculo de Gradientes

Para optimizar lso pesos de los parametros de la red, necesitaremos calcular las derivadas de nuestra funcion de perdida respecto a los parametros, necesitamos \frac{\partial loss}{\partial w} y
\frac{\partial loss}{\partial b} para valores fijos de `x` y `y`. Para calcular
esas derivadas, llamamos a `loss.backward()` y luego obtenemos los valores desde
`w.grad` y `b.grad`:

In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

## Desactivar el seguimeinto de gradientes

Por defecto los tensores con reequires_grad=True registran su historia computacional y soprtan el calculo de gradientes, hay coas en lso que no necestiamos eso, como por ejemplo cuando queremos aplicar algunos datos de entrada y solo queremos hacer calculos de forward a traves de la red, podemos dejar de hacer calculos con un bloque torch.no_grad(), otra manera de conseguir el mismo resultado es usando el metodo detach() en el tensor.

In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)
#------
z = torch.matmul(x,w)+b
z_det = z.detach()
print(z_det.requires_grad)